# Vocabulary & Data Preparation


Convert text → token IDs

Add <sos> and <eos>

Prepare tensors

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# -----------------------------
# Vocabulary
# -----------------------------
vocab = ["<pad>", "<sos>", "<eos>", "i", "like", "ai"]

# Create mappings
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

def encode(sentence):
    return [word2idx[w] for w in sentence]

def decode(indices):
    return [idx2word[i] for i in indices]

# Example data
input_sentence = ["i", "like", "ai"]
target_sentence = ["ai", "like", "i"]

# Convert to tensors
src = torch.tensor([encode(input_sentence)])

# Add <sos> and <eos>
trg = torch.tensor([
    [word2idx["<sos>"]] + encode(target_sentence) + [word2idx["<eos>"]]
])

# Positional Encoding

Transformers have:

 No recurrence

 No inherent order awareness

So we inject position information manually

**Key Idea**

Even positions → sine

Odd positions → cosine

In [ ]:
# -----------------------------
# Positional Encoding
# -----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=100):
        super().__init__()

        # Create matrix [max_len × embed_size]
        pe = torch.zeros(max_len, embed_size)

        for pos in range(max_len):
            for i in range(0, embed_size, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (i / embed_size)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** (i / embed_size)))

        # Add batch dimension
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        # Add positional encoding to embeddings
        return x + self.pe[:, :x.size(1)].to(x.device)

# Transformer Model

Replace RNN/LSTM completely

Use self-attention + feedforward layers

**Key Components**

Embedding

Positional encoding

Transformer (encoder + decoder)

Linear output layer

In [ ]:
# -----------------------------
# Transformer Model
# -----------------------------
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, num_layers):
        super().__init__()

        # Token embeddings
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(embed_size)

        # Transformer block (encoder + decoder)
        self.transformer = nn.Transformer(
            d_model=embed_size,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            batch_first=True
        )

        # Output projection
        self.fc = nn.Linear(embed_size, vocab_size)

    def forward(self, src, trg):
        # Step 1: Embed + positional encoding
        src_emb = self.pos_encoding(self.embedding(src))
        trg_emb = self.pos_encoding(self.embedding(trg))

        # Step 2: Mask future tokens (very important!)
        tgt_mask = self.transformer.generate_square_subsequent_mask(
            trg.size(1)
        ).to(trg.device)

        # Step 3: Transformer forward pass
        out = self.transformer(src_emb, trg_emb, tgt_mask=tgt_mask)

        # Step 4: Map to vocabulary
        return self.fc(out)

# Training Setup


Define architecture size

Set optimizer and loss

In [ ]:
# -----------------------------
# Training Setup
# -----------------------------
vocab_size = len(vocab)
embed_size = 32
num_heads = 2
num_layers = 2

model = TransformerModel(vocab_size, embed_size, num_heads, num_layers)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop

 We use:

Input: trg[:, :-1]

Target: trg[:, 1:]

This is shifted decoding

In [ ]:
# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(300):
    optimizer.zero_grad()

    # Forward pass
    output = model(src, trg[:, :-1])

    # Reshape for loss
    output = output.reshape(-1, vocab_size)
    target = trg[:, 1:].reshape(-1)

    # Compute loss
    loss = criterion(output, target)

    # Backpropagation
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 2.5482
Epoch 50, Loss: 0.0148
Epoch 100, Loss: 2.4741
Epoch 150, Loss: 0.3549
Epoch 200, Loss: 0.3151
Epoch 250, Loss: 0.2844


# Inference (Autoregressive Generation)


Generate tokens step-by-step

No teacher forcing

In [ ]:
# -----------------------------
# Inference
# -----------------------------
def predict(model, src, max_len=10):
    model.eval()

    # Start with <sos>
    trg_input = torch.tensor([[word2idx["<sos>"]]])

    with torch.no_grad():
        for _ in range(max_len):
            output = model(src, trg_input)

            # Take last token prediction
            next_token = output[:, -1, :].argmax(1).item()

            # Stop if <eos>
            if next_token == word2idx["<eos>"]:
                break

            # Append token
            trg_input = torch.cat(
                [trg_input, torch.tensor([[next_token]])],
                dim=1
            )

    return decode(trg_input.squeeze().tolist()[1:])

print("Prediction:", predict(model, src))

Prediction: ['ai', 'ai', 'like', 'i']
